Test Case 1 - Are the tables there?:  
* Tables in synapse 
* ODW Source == odw_base
* Manually checking the table names 

Test Case 2:  Are the views there?:
* Schema by schema 
* Checking counts and then manually looking missing views when counts mismatch.

Test Case 3:  Comparing Column Names and Data Types On the Views:
* Manually checking Views by writing a query in synapse and then reviewing the 

Test Case 4:  Check the view definition 
* Query in Synapse followed by the 

Test Case 5:  Check the table counts for the records
* Running a count query in both systems one by one and comparing manually 

In [0]:
DECLARE OR REPLACE VARIABLE target_catalog STRING DEFAULT "main";
DECLARE OR REPLACE VARIABLE target_schema STRING DEFAULT "default";
DECLARE OR REPLACE VARIABLE source_database STRING DEFAULT "odb";
DECLARE OR REPLACE VARIABLE source_schema STRING DEFAULT "";

In [0]:
SELECT target_catalog, target_schema, source_database, source_schema;

In [0]:
SET VARIABLE target_catalog = :target_catalog;
SET VARIABLE target_schema = :target_schema;
SET VARIABLE source_database = :source_database;
SET VARIABLE source_schema = :source_schema;

In [0]:
SELECT target_catalog, target_schema, source_database, source_schema;

In [0]:
select * from mgiglia.information_schema.tables limit 10;

In [0]:
-- select distinct Source_Name, object_name 
-- from etl.vAllObjectLoadMetadata where Source_Name= 'gis' --'Qnxt_Plandata_rpt'  --'' --'' --'' 
-- and active_flag =1 --21
-- --and object_name like '%odw%'
-- order by 1,2


-- this is the SQL that you were running in Synapse to get table names, or column names, etc. 
DECLARE OR REPLACE VARIABLE source_test_stmnt STRING;

SET VARIABLE source_test_stmnt = "(
    SELECT 
      Source_Name as table_catalog
      ,'etl' as table_schema
      ,object_name as table_name 
    FROM etl.vAllObjectLoadMetadata
    WHERE 
      Source_Name = '" || source_database || "'
    ORDER BY 
      1, 2;
  ) q";

SELECT source_test_stmnt;

In [0]:
DECLARE OR REPLACE VARIABLE source_view_stmnt STRING DEFAULT "
CREATE OR REPLACE TEMPORARY VIEW source_check AS
SELECT *
FROM jdbc.`<synapse_database_url>` -- secret('jdbc_connection_details', 'jdbc_url')
OPTIONS (
  user = secret('jdbc_connection_details', 'jdbc_user_name'),
  password = secret('jdbc_connection_details', 'jdbc_password'),
  dbtable = '" || source_test_stmnt || "'
);
";

SELECT source_view_stmnt;

In [0]:
-- Demo only since the above SQL won't work without an actual JDBC connection for Spark to make.  
DECLARE OR REPLACE VARIABLE source_view_stmnt STRING DEFAULT "
CREATE OR REPLACE TEMPORARY VIEW source_check AS
SELECT * FROM " || target_catalog || ".information_schema.tables;";

SELECT source_view_stmnt;

In [0]:
EXECUTE IMMEDIATE source_view_stmnt;

In [0]:
select * from source_check;

In [0]:
-- the actual check is a minus query between the view you created against synapse, and the information schema in Unity Catalog
SELECT * from source_check 
EXCEPT ALL
SELECT * FROM mgiglia.information_schema.tables
WHERE schema = target_schema; 

In [0]:
CREATE OR REPLACE TEMPORARY VIEW source_tables AS
SELECT *
FROM jdbc.`<synapse_database_url>` -- secret("jdbc_connection_details", "jdbc_url")
OPTIONS (
  user = secret("jdbc_connection_details", "jdbc_user_name"),
  password = secret("jdbc_connection_details", "jdbc_password"),
  dbtable = '<table_name>'
);